In [3]:
import openmc
import openmc.mgxs as mgxs
import os
import glob
import json

# ==============================================================================
# 0. CLEANUP
# ==============================================================================
for file in glob.glob("*.h5"):
    try:
        os.remove(file)
    except:
        pass

os.environ['OPENMC_CROSS_SECTIONS'] = '/home/jovyan/work/Beavers/Pincell/endfb/cross_sections.xml'
# ==============================================================================
# 1. SETUP (BEAVRS HZP Pin Cell)
# ==============================================================================
# HZP Temperature in Kelvin
T_HZP = 600.0

# Fuel: UO2 at 3.1 wt% enrichment
fuel = openmc.Material(name='UO2_3.1')
fuel.set_density('g/cm3', 10.24)
fuel.add_element('U', 1.0, enrichment=3.1)
fuel.add_element('O', 2.0)
fuel.temperature = T_HZP

# Gap: Helium
gap = openmc.Material(name='Helium_Gap')
gap.set_density('g/cm3', 0.0015)
gap.add_element('He', 1.0)
gap.temperature = T_HZP

# Cladding: Zircaloy 4
clad = openmc.Material(name='Zircaloy')
clad.set_density('g/cm3', 6.55)
clad.add_element('Zr', 0.98115, percent_type='wo')
clad.add_element('Sn', 0.01450, percent_type='wo')
clad.add_element('Fe', 0.00210, percent_type='wo')
clad.add_element('Cr', 0.00100, percent_type='wo')
clad.temperature = T_HZP

# Moderator: Borated Water (975 ppm)
water = openmc.Material(name='Borated_Water')
water.set_density('g/cm3', 0.743)
water.add_element('H', 2.0)
water.add_element('O', 1.0)
water.add_element('B', 9.75e-4)
try:
    water.add_s_alpha_beta('c_H_in_H2O')
except:
    pass
water.temperature = T_HZP

materials = openmc.Materials([fuel, gap, clad, water])

# BEAVRS Geometry Definitions
pitch = 1.25984
r_pellet = 0.39218
r_inner_clad = 0.40005
r_outer_clad = 0.45720

pellet_surf = openmc.ZCylinder(r=r_pellet)
inner_clad_surf = openmc.ZCylinder(r=r_inner_clad)
outer_clad_surf = openmc.ZCylinder(r=r_outer_clad)

left = openmc.XPlane(x0=-pitch/2, boundary_type='reflective')
right = openmc.XPlane(x0=pitch/2, boundary_type='reflective')
bottom = openmc.YPlane(y0=-pitch/2, boundary_type='reflective')
top = openmc.YPlane(y0=pitch/2, boundary_type='reflective')

fuel_cell = openmc.Cell(fill=fuel, region=-pellet_surf)
gap_cell = openmc.Cell(fill=gap, region=+pellet_surf & -inner_clad_surf)
clad_cell = openmc.Cell(fill=clad, region=+inner_clad_surf & -outer_clad_surf)
water_cell = openmc.Cell(fill=water, region=+outer_clad_surf & +left & -right & +bottom & -top)

pin_universe = openmc.Universe(cells=(fuel_cell, gap_cell, clad_cell, water_cell))
geometry = openmc.Geometry(pin_universe)

# Simulation Settings
settings = openmc.Settings()
settings.batches = 100
settings.inactive = 20
settings.particles = 10000
settings.temperature = {'method': 'nearest'}
bounds = [-r_pellet, -r_pellet, -1, r_pellet, r_pellet, 1]
uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:], only_fissionable=True)
settings.source = openmc.IndependentSource(space=uniform_dist)

# ==============================================================================
# 2. MGXS SETUP
# ==============================================================================
groups = mgxs.EnergyGroups([0.0, 0.625, 20.0e6])
domain = pin_universe

# Standard Library
lib = mgxs.Library(geometry)
lib.energy_groups = groups
lib.domain_type = 'universe'
lib.domains = [domain]
lib.mgxs_types = ['transport', 'nu-fission', 'absorption', 'scatter matrix']
lib.build_library()

# Kinetics Data
beta_xs = mgxs.Beta(domain=domain, energy_groups=groups)
beta_xs.delayed_groups = list(range(1, 7))
beta_xs.estimator = 'analog'

decay_xs = mgxs.DecayRate(domain=domain, energy_groups=groups)
decay_xs.delayed_groups = list(range(1, 7))
decay_xs.estimator = 'analog'

inv_vel = mgxs.InverseVelocity(domain=domain, energy_groups=groups)

# Combine and Run
tallies = openmc.Tallies()
lib.add_to_tallies_file(tallies, merge=True)
for tally in beta_xs.tallies.values():
    tallies.append(tally, merge=True)
for tally in decay_xs.tallies.values():
    tallies.append(tally, merge=True)
for tally in inv_vel.tallies.values():
    tallies.append(tally, merge=True)

model = openmc.Model(geometry=geometry, materials=materials, settings=settings, tallies=tallies)
print("Running OpenMC BEAVRS Model...")
sp_file = model.run()

# ==============================================================================
# 3. EXTRACT AND WRITE MOLTRES JSON
# ==============================================================================
sp = openmc.StatePoint(sp_file)
keff_mc = sp.k_combined.nominal_value

# Load data into objects
lib.load_from_statepoint(sp)
beta_xs.load_from_statepoint(sp)
decay_xs.load_from_statepoint(sp)
inv_vel.load_from_statepoint(sp)

u = pin_universe

# Get base physics arrays
trans_xs = lib.get_mgxs(u, 'transport').get_pandas_dataframe()['mean'].values
abs_xs = lib.get_mgxs(u, 'absorption').get_pandas_dataframe()['mean'].values
nusigf_xs = lib.get_mgxs(u, 'nu-fission').get_pandas_dataframe()['mean'].values
inv_vel_values = inv_vel.get_pandas_dataframe()['mean'].values

# Calculate Scatter Matrix Elements (1->1, 1->2, 2->1, 2->2)
scatter = lib.get_mgxs(u, 'scatter matrix')
df = scatter.get_pandas_dataframe()
s_11 = df[(df['group in'] == 1) & (df['group out'] == 1)]['mean'].values[0]
s_12 = df[(df['group in'] == 1) & (df['group out'] == 2)]['mean'].values[0]
s_21 = df[(df['group in'] == 2) & (df['group out'] == 1)]['mean'].values[0]
s_22 = df[(df['group in'] == 2) & (df['group out'] == 2)]['mean'].values[0]

# Precursors
betas = beta_xs.get_pandas_dataframe()['mean'].values
lambdas = decay_xs.get_pandas_dataframe()['mean'].values

# Format for Moltres JSON
xs_data = {
    "fuel": {
        "566.0": {
            "G": 2,
            "DIFF": [1.0 / (3.0 * trans_xs[0]), 1.0 / (3.0 * trans_xs[1])],
            # Normalize fission source by k_eff to start critical
            "NSF": [(nusigf_xs[0] / keff_mc), (nusigf_xs[1] / keff_mc)],
            "REMXS": [abs_xs[0] + s_12, abs_xs[1] + s_21],
            "SCAT": [
                [0.0, s_21], # Scatter into group 1 (from 1, from 2)
                [s_12, 0.0]  # Scatter into group 2 (from 1, from 2)
            ],
            "RECIPVEL": [inv_vel_values[0], inv_vel_values[1]],
            "BETA_EFF": betas.tolist(),
            "LAMBDA": lambdas.tolist()
        }
    }
}

with open("xsdata.json", "w") as f:
    json.dump(xs_data, f, indent=4)

print(f"\nSUCCESS! xsdata.json created.")
print(f"OpenMC k_eff: {keff_mc:.5f}")
sp.close()

/openmc_venv/lib/python3.11/site-packages/openmc/stats/multivariate.py:829: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=39.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=40.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=41.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=42.
  warn(msg, IDWarning)
/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=53.
  warn(msg, IDWarning)


Running OpenMC BEAVRS Model...
                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
       

/openmc_venv/lib/python3.11/site-packages/openmc/statepoint.py:277: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(
